In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/archive.zip" -d "/content/dataset"

In [ ]:
import os

print(os.listdir("/content/dataset"))

['myntradataset', 'images', 'styles.csv']


In [ ]:
import pandas as pd

df = pd.read_csv("/content/dataset/styles.csv", on_bad_lines="skip")

print(df.shape)
df.head()

(44424, 10)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [ ]:
men_tshirt_df = df[
    (df["gender"] == "Men") &
    (df["articleType"] == "Tshirts")
].copy()

print("Total Men's T-shirts:", len(men_tshirt_df))

Total Men's T-shirts: 5243


In [ ]:
import os

IMAGE_DIR = "/content/dataset/images"

men_tshirt_df["image_path"] = men_tshirt_df["id"].apply(
    lambda x: os.path.join(IMAGE_DIR, str(x) + ".jpg")
)

men_tshirt_df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,/content/dataset/images/53759.jpg
5,1855,Men,Apparel,Topwear,Tshirts,Grey,Summer,2011.0,Casual,Inkfruit Mens Chain Reaction T-shirt,/content/dataset/images/1855.jpg
27,7990,Men,Apparel,Topwear,Tshirts,Navy Blue,Fall,2011.0,Sports,Fila Men's Round Neck Navy Blue T-shirt,/content/dataset/images/7990.jpg
55,5891,Men,Apparel,Topwear,Tshirts,Black,Summer,2011.0,Casual,Puma Men's Stripe Polo Black T-shirt,/content/dataset/images/5891.jpg
59,10866,Men,Apparel,Topwear,Tshirts,Red,Fall,2011.0,Casual,Wrangler Men Motor Rider Red T-Shirts,/content/dataset/images/10866.jpg


In [ ]:
men_tshirt_df["exists"] = men_tshirt_df["image_path"].apply(os.path.exists)

print(men_tshirt_df["exists"].value_counts())

exists
True     5242
False       1
Name: count, dtype: int64


In [ ]:
men_tshirt_df = men_tshirt_df[men_tshirt_df["exists"] == True].copy()
men_tshirt_df = men_tshirt_df.drop(columns=["exists"])

print("Final Men's T-shirt images:", len(men_tshirt_df))

Final Men's T-shirt images: 5242


In [ ]:
men_tshirt_df.to_csv("/content/drive/MyDrive/mens_tshirt_dataframe.csv", index=False)

print("Clean DataFrame saved successfully")

Clean DataFrame saved successfully


In [ ]:
from sklearn.model_selection import train_test_split

historical_df, new_market_df = train_test_split(
    men_tshirt_df,
    train_size=3145,
    random_state=42
)

print("Historical:", len(historical_df))
print("New Market:", len(new_market_df))

Historical: 3145
New Market: 2097


In [ ]:
# Required count as per scaled PS
required_preferred = 1468
required_not_preferred = 1677

# Preferred colors
preferred_colours = ["Black", "Grey", "Navy Blue", "White", "Blue"]

preferred_pool = historical_df[
    historical_df["baseColour"].isin(preferred_colours)
].copy()

not_preferred_pool = historical_df[
    ~historical_df["baseColour"].isin(preferred_colours)
].copy()

# Preferred final
preferred_final = preferred_pool.sample(n=required_preferred, random_state=42)

# Not preferred final
remaining_needed = required_not_preferred - len(not_preferred_pool)

extra_not_preferred = preferred_pool.drop(preferred_final.index).sample(
    n=remaining_needed,
    random_state=42
)

not_preferred_final = pd.concat(
    [not_preferred_pool, extra_not_preferred],
    ignore_index=True
)

# Labels
preferred_final["label"] = 1
not_preferred_final["label"] = 0

# Final preference dataset
preference_df = pd.concat(
    [preferred_final, not_preferred_final],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print(preference_df["label"].value_counts())
print("Final Historical Dataset:", len(preference_df))

label
0    1677
1    1468
Name: count, dtype: int64
Final Historical Dataset: 3145
